# Phase 6 — Deployment

Phase 5 produced `xgb_v2_calibrated` and recommended an operating threshold of `t = 0.13`. Phase 6 wraps that model in the three pieces a credit-risk team actually needs to run it:

1. **Online inference** — `src/serve.py` (FastAPI) for single-application scoring at decision time.
2. **Batch scoring** — `src/score_batch.py` (CLI) for nightly reruns over the application pipeline.
3. **Monitoring** — `src/monitor.py` for population-stability and prediction-drift signals.

Each piece reads the *same* model artifact and the *same* training-set categorical levels, so a single retraining run can update all three by replacing the joblib file.

## 1. Architecture in one diagram

```
  Loan application (JSON)                    Application pipeline (parquet)
          │                                          │
          ▼                                          ▼
  POST /predict                              python score_batch.py
          │                                          │
          └────────────┬─────────────────────────────┘
                       ▼
             xgb_v2_calibrated.joblib
             (XGBoost + isotonic)
                       │
                       ▼
         p_default, decision (t=0.13)
                       │
                       ▼
         outputs/logs/predictions.jsonl
                       │
                       ▼
             python monitor.py
             (daily: PSI, reject rate)
```

All three entry-points share `prepare.CATEGORICAL_COLS` + `models.split_xy()` so feature ordering and categorical levels stay in lock-step with training.

## 2. Demo the online endpoint via TestClient

We don't need uvicorn for this — FastAPI's `TestClient` exercises the same handlers in-process. Real production runs `python serve.py` (binds 127.0.0.1:8000) behind an API gateway.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from fastapi.testclient import TestClient

PROJECT = Path(r'C:\Users\User\Desktop\Credit Risk Prediction System')
sys.path.insert(0, str(PROJECT / 'src'))

from prepare import load_processed, CATEGORICAL_COLS
import serve as serve_module
import monitor

In [ ]:
def row_to_payload(row):
    """Convert a prepared test row into a LoanApplication-shaped dict."""
    payload = {}
    for key, val in row.items():
        if key in ('default', 'issue_year'):
            continue
        if pd.isna(val):
            payload[key] = None
        elif key in CATEGORICAL_COLS:
            payload[key] = str(val)
        elif key in ('term', 'emp_length_missing', 'pub_rec_bankruptcies',
                     'mort_acc', 'credit_history_years'):
            payload[key] = int(val)
        else:
            payload[key] = float(val)
    return payload

_, test = load_processed()
sample = test.sample(n=10, random_state=42).reset_index(drop=True)
payloads = [row_to_payload(sample.iloc[i]) for i in range(len(sample))]
payloads[0]

In [ ]:
with TestClient(serve_module.app) as client:
    health = client.get('/health').json()
    print('Health:', health)

    print('\n/predict on three sample applications:')
    for i in range(3):
        r = client.post('/predict', json=payloads[i]).json()
        print(f'  app {i}  actual={sample.iloc[i]["default"]}  '
              f'p_default={r["p_default"]:.4f}  decision={r["decision"]}')

    print('\n/predict/batch on 10 applications:')
    batch = client.post('/predict/batch', json={'applications': payloads}).json()
    summary = pd.DataFrame([{**p, 'actual': sample.iloc[i]['default']}
                            for i, p in enumerate(batch['predictions'])])
    print(summary.to_string(index=False))

**Reading the output.** Some `actual=0` rows get rejected — that's the cost-of-FN-is-5x-cost-of-FP doing its job. False rejects of good borrowers are the price of catching defaulters; the threshold was tuned exactly to minimize that combined cost.

### Input validation

Pydantic rejects malformed payloads with HTTP 422 *before* they touch the model — silently coercing a missing field to NaN would let bad data into the prediction log.

In [ ]:
with TestClient(serve_module.app) as client:
    bad = dict(payloads[0])
    del bad['loan_amnt']
    r = client.post('/predict', json=bad)
    print(f'status={r.status_code}')
    print(r.json())

## 3. Batch scoring CLI

For pipelines that load applications from a warehouse on a schedule, the CLI is faster than HTTP and cheaper to operate:

```bash
python src/score_batch.py \
    --input  data/processed/test.parquet \
    --output outputs/scored_test.parquet \
    --threshold 0.13
```

1,000 rows score in ~1 second on a laptop; the full 244K test set would run in well under a minute.

## 4. Monitoring

Every `/predict` and `/predict/batch` call appends to `outputs/logs/predictions.jsonl`. `monitor.py` reads this log, scores the same train set as the reference distribution, and reports:

- **PSI** between production scores and the training reference. <0.10 stable, 0.10-0.25 watch, ≥0.25 retrain.
- **Daily mean p_default** — climbing values point to a real shift in the application population.
- **Reject rate** — confirms the threshold is doing what the cost-curve said it would.

PSI is suppressed below 1,000 samples (small N would explode the log-ratio on empty quantile bins).

In [ ]:
preds = monitor.load_predictions()
print(f'Logged predictions: {len(preds)}')
if len(preds):
    ref = monitor.reference_train_scores()
    r = monitor.report(preds, ref)
    monitor.print_report(r)

## 5. Operational playbook (one page)

### Promoting a new model
1. Run `notebook 05_evaluation.ipynb` end-to-end. Confirm test KS ≥ Phase 5 baseline (0.29).
2. `joblib.dump(...)` to `outputs/models/xgb_vN_calibrated.joblib`.
3. Set `MODEL_NAME=xgb_vN_calibrated` env var and restart `serve.py`.
4. Run a shadow window (route 5% of traffic, compare PSI vs current model).
5. Cut over.

### Threshold change
Set `DECISION_THRESHOLD` env var. No retraining needed. Re-run the cost-curve in notebook 05 with the new cost ratio first.

### Drift response
- PSI in [0.10, 0.25): investigate. Check feature-level drift (per-column histograms vs training).
- PSI ≥ 0.25 sustained over 7 days: retrain. The most recent year of labeled data becomes the new calibration slice.

### Retraining cadence
Quarterly is the default — short enough to absorb macro shifts, long enough to gather enough labeled outcomes (loans take 36-60 months to mature, but Late-16-30 events surface in weeks).

### Known limitations of v2 to flag for stakeholders
1. **Decile-1 lift = 1.93×** — operationally useful but the top-decile capture is only 19%. A specialist would expect 30%+ on well-tuned credit models. Phase 7 hyperparameter sweep is open work.
2. **2018 horizon survivorship bias** — the test set's 27% default rate is a lower bound. Real long-run defaults will be higher.
3. **Class weights de-calibrate; isotonic recovers** — but the recovery depends on the calibration slice resembling production. If application mix shifts (e.g., new product line), recalibrate before deploying.

## 6. Phase-by-phase summary (handed off to the next person)

| Phase | Deliverable | Headline output |
|---|---|---|
| 1 — Business understanding | Project structure + load helpers | `src/load.py`, 26 application-time features |
| 2 — Data understanding | `02_data_understanding.ipynb` | 1.37M rows, 21.48% default, imbalance 3.7:1 |
| 3 — Data preparation | `03_data_preparation.ipynb`, `prepare.py` | Time split: 1.13M train (20.25%) / 244K test (27.20%) |
| 4 — Modeling | `04_modeling.ipynb`, `models.py` | XGBoost test ROC-AUC 0.71, KS 0.30 |
| 5 — Evaluation | `05_evaluation.ipynb`, `evaluate.py` | Calibrated XGB, threshold t=0.13, PSI=0.01 |
| 6 — Deployment | `06_deployment.ipynb`, `serve.py`, `score_batch.py`, `monitor.py` | FastAPI + CLI + monitoring scaffolding |

**The chain is reproducible end-to-end.** Drop the raw CSV in place and run smokes 2 through 6 in order; every artifact downstream of `accepted_2007_to_2018Q4.csv` regenerates.

### Check-for-understanding

1. The service pins categorical levels at startup from the training parquet. What goes wrong if a new `addr_state` shows up in production that wasn't in training, and how does the current code handle it?
2. PSI is suppressed below 1,000 samples. Why is computing it on, say, 50 predictions worse than not reporting it at all?
3. The `/predict` log records the *input features* alongside the score. Why is that more useful for drift detection than just logging the score?
4. The notebook recommends quarterly retraining. Sketch the worst case where quarterly is **too slow**.
5. If the bank changes its cost ratio from 5:1 to 10:1, do we need to retrain, recalibrate, or just change a flag? Why?